In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [2]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
num_epochs=4
batch_size=4
learning_rate=0.001

In [6]:
transform=transforms.Compose([transforms.ToTensor(),
transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])

train_dataset=torchvision.datasets.CIFAR10(root='/data',train=True,download=True,transform=transform)

test_dataset=torchvision.datasets.CIFAR10(root='/data',train=False,download=True,transform=transform)

train_loader=torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

test_loader=torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=True)

classes=(
    'plane','car','bird','cat','deer','dog','frog','horse','ship','truck'
)


100.0%
d:\Projects\AI-pipelines\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [13]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet,self).__init__()
        self.conv1=nn.Conv2d(3,6,5)
        self.pool=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(6,16,5)
        self.fc1=nn.Linear(16*5*5,120)
        self.fc2=nn.Linear(120,84)
        self.fc3=nn.Linear(84,10)

    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=x.view(-1,16*5*5)
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)

        return x

In [14]:
model = ConvNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

n_total_steps = len(train_loader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # origin shape: [4, 3, 32, 32] = 4, 3, 1024
        # input_layer: 3 input channels, 6 output channels, 5 kernel size
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 2000 == 0:
            print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{n_total_steps}], Loss: {loss.item():.4f}')

Epoch [1/4], Step [2000/12500], Loss: 2.2810
Epoch [1/4], Step [4000/12500], Loss: 2.2901
Epoch [1/4], Step [6000/12500], Loss: 2.2846
Epoch [1/4], Step [8000/12500], Loss: 2.3749
Epoch [1/4], Step [10000/12500], Loss: 2.0504
Epoch [1/4], Step [12000/12500], Loss: 2.0658
Epoch [2/4], Step [2000/12500], Loss: 1.5646
Epoch [2/4], Step [4000/12500], Loss: 2.0062
Epoch [2/4], Step [6000/12500], Loss: 1.4932
Epoch [2/4], Step [8000/12500], Loss: 1.7183
Epoch [2/4], Step [10000/12500], Loss: 1.3268
Epoch [2/4], Step [12000/12500], Loss: 2.3726
Epoch [3/4], Step [2000/12500], Loss: 1.3391
Epoch [3/4], Step [4000/12500], Loss: 1.8318
Epoch [3/4], Step [6000/12500], Loss: 1.5685
Epoch [3/4], Step [8000/12500], Loss: 0.8648
Epoch [3/4], Step [10000/12500], Loss: 2.0207
Epoch [3/4], Step [12000/12500], Loss: 1.1415
Epoch [4/4], Step [2000/12500], Loss: 0.9929
Epoch [4/4], Step [4000/12500], Loss: 1.6721
Epoch [4/4], Step [6000/12500], Loss: 1.9794
Epoch [4/4], Step [8000/12500], Loss: 1.2071
Epoc

In [15]:
with torch.no_grad():
    n_correct = 0
    n_samples = 0
    n_class_correct = [0 for i in range(10)]
    n_class_samples = [0 for i in range(10)]
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        # max returns (value ,index)
        _, predicted = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()
        
        for i in range(batch_size):
            label = labels[i]
            pred = predicted[i]
            if (label == pred):
                n_class_correct[label] += 1
            n_class_samples[label] += 1

    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy of the network: {acc} %')

    for i in range(10):
        acc = 100.0 * n_class_correct[i] / n_class_samples[i]
        print(f'Accuracy of {classes[i]}: {acc} %')


Accuracy of the network: 45.59 %
Accuracy of plane: 45.1 %
Accuracy of car: 66.7 %
Accuracy of bird: 19.7 %
Accuracy of cat: 24.9 %
Accuracy of deer: 18.8 %
Accuracy of dog: 51.3 %
Accuracy of frog: 56.7 %
Accuracy of horse: 58.0 %
Accuracy of ship: 58.9 %
Accuracy of truck: 55.8 %
